# 无评分实验 - 追踪一个 RAG 系统

---

欢迎来到使用 Weaviate 和 Phoenix 进行 RAG 系统追踪与评估的无评分实验！在本次互动实验中，你将学习如何有效使用遥测来追踪并排查 RAG 系统问题。你还将理解并实践一些关键概念，包括 span、trace 和 chain，它们对于监控与提升系统性能非常重要。

在本实验中，你将：

- 理解如何设置并使用遥测来监控你的 RAG 系统。
- 学习 trace 与 span 的概念。
- 通过查看 trace 来理解系统流程中的完整路径与交互。
- 使用 [Phoenix Arize](https://phoenix.arize.com/) 工具对遥测数据进行可视化与分析。
- 观察一个结合 Phoenix 与 Weaviate 的小型 RAG 流水线示例。
  

---

<h4 style="color:black; font-weight:bold;">使用目录（TABLE OF CONTENTS）</h4>
JupyterLab 为你提供了便捷的作业导航方式。目录位于左侧面板中的“目录（Table of Contents）”选项卡，如下图所示。

![TOC Location](images/toc.png)

---


# 目录
- [ 1 - 介绍](#1)
  - [ 1.1 导入必要库](#1-1)
- [ 2 - 遥测快速入门](#2)
  - [ 2.1 Span](#2-1)
- [ 3 - 使用 Phoenix 进行遥测](#3)
  - [ 3.1 启动 Phoenix 应用](#3-1)
  - [ 3.2 准备遥测配置](#3-2)
  - [ 3.3 运行流水线](#3-3)
  - [ 3.4 Chain](#3-4)
  - [ 3.5 使用 UI 分析 Trace](#3-5)
- [ 4 - 结合 Weaviate 的追踪与评估](#4)
  - [ 4.1 配置追踪器](#4-1)
  - [ 4.2 准备 Weaviate 客户端与集合](#4-2)
  - [ 4.3 检索器](#4-3)
  - [ 4.4 使用 `openai` 库进行 LLM 调用](#4-4)
- [ 5 - 评估一个 RAG 系统](#5)


<a id='1'></a>
## 1 - 介绍

---
在 RAG 系统中，遥测是监控与优化性能的关键。通过采集并传输系统运行数据（例如 span 表示单步操作、trace 表示完整流程），遥测让我们可以观察系统如何检索、处理与生成信息。这种可见性有助于识别瓶颈、定位问题，并提升系统效率。

<a id='1-1'></a>
### 1.1 导入必要库


In [1]:
import utils
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.trace import Status, StatusCode
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
import warnings
warnings.filterwarnings("ignore")

Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 30] Read-only file system: '.models/models--BAAI--bge-base-en-v1.5/.no_exist/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/adapter_config.json'


<a id='2'></a>
## 2 - 遥测快速入门
---
<a id='2-1'></a>
### 2.1 Span

在遥测中，span 表示系统中的一个独立操作或任务。你可以把它理解为某个具体动作的快照，记录其开始与结束时间。span 还会包含任务行为和关键事件等细节。通过追踪 span，你可以了解操作耗时并发现潜在问题，从而更好地理解并改进系统性能。

下面看一个如何使用 [OpenTelemetry](https://opentelemetry.io/) 配置简单 tracer 的示例，Phoenix 也使用这个工具。

**注意：** 这里我们创建的是本地 tracer provider（不是全局 provider），这样可以演示 OpenTelemetry 概念，同时不影响后续将使用的 Phoenix tracer provider。

In [ ]:
# 定义描述应用的资源属性
# 这里设置服务名，用于标识被追踪的对象
resource = Resource(attributes={
    "service.name": "Test Service"
})

# 为演示目的设置一个本地（非全局）tracer provider
# 这样可以展示 OpenTelemetry 概念，同时不阻塞后续 Phoenix 的全局 provider
local_tracer_provider = TracerProvider(resource=resource)

# 创建控制台导出器，用于把 span 输出到控制台进行演示
# 在真实场景中，你通常会使用 OTLP 导出器发送到追踪系统
console_exporter = ConsoleSpanExporter()

# 设置 span 处理器
# SimpleSpanProcessor 会在每个 span 结束后立即发送到导出器
span_processor = SimpleSpanProcessor(console_exporter)

# 将 span 处理器添加到本地 tracer provider
local_tracer_provider.add_span_processor(span_processor)

# 从本地 provider 获取 tracer，用于创建和管理 span
tracer = local_tracer_provider.get_tracer(__name__)

#### 2.1.1 一个玩具版检索函数

这是一个基础函数，用于演示如何使用 span 为文档检索操作配置追踪。

In [ ]:
def retrieve(query, fail=False):
    # 启动一个 span 来追踪检索流程
    with tracer.start_as_current_span("retrieving_documents") as span:
        # 记录开始检索事件
        span.add_event("Starting retrieve")
        # 将输入 query 作为属性记录，便于观测
        span.set_attribute("input.query", query)
        try:
            # 如果 'fail' 为 True，则模拟检索失败
            if fail:
                raise ValueError(f"Retrieve failed for query: {query}")

            # 模拟检索到的文档列表
            retrieved_docs = ['retrieved doc1', 'retrieved doc2', 'retrieved doc3']
            # 记录每个检索文档的细节
            for i, doc in enumerate(retrieved_docs):
                span.set_attribute(f"retrieval.documents.{i}.document.id", i)
                span.set_attribute(f"retrieval.documents.{i}.document.content", doc)
                span.set_attribute(f"retrieval.documents.{i}.document.metadata", f"Metadata for document {i}")
        except Exception as e:
            # 若发生异常，记录并将 span 状态标记为错误
            span.set_status(Status(StatusCode.ERROR, str(e)))
            span.set_attribute("error.type", type(e).__name__)
            span.set_attribute("error.message", str(e))
            # 将异常继续抛出，由调用方处理
            raise

        # 若无错误，标记 span 成功
        span.set_status(Status(StatusCode.OK))
        return retrieved_docs

In [ ]:
# tracer 已配置，可在输出中看到 span
retrieve("Test")

{
    "name": "retrieving_documents",
    "context": {
        "trace_id": "0xe99f94390c665a081d194aa3a27d3168",
        "span_id": "0x18b8ff8f340913ac",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-03T12:18:22.591494Z",
    "end_time": "2026-04-03T12:18:22.591586Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "input.query": "Test",
        "retrieval.documents.0.document.id": 0,
        "retrieval.documents.0.document.content": "retrieved doc1",
        "retrieval.documents.0.document.metadata": "Metadata for document 0",
        "retrieval.documents.1.document.id": 1,
        "retrieval.documents.1.document.content": "retrieved doc2",
        "retrieval.documents.1.document.metadata": "Metadata for document 1",
        "retrieval.documents.2.document.id": 2,
        "retrieval.documents.2.document.content": "retrieved doc3",
        "retrieval.documents.2.document.metadata": "M

['retrieved doc1', 'retrieved doc2', 'retrieved doc3']

## 2.2 Trace
---
Trace 是一组 span 的集合，用于表示一次请求或事务在系统多个组件中的完整流转路径。也就是说，一个 trace 由围绕同一任务的多个相关 span 组成。

下面我们补全一个 toy RAG 流水线，看看 trace 的样子。

In [ ]:
def format_documents(retrieved_docs):
    # 启动 span，追踪文档格式化过程
    with tracer.start_as_current_span("call_format_documents") as span:
        # 记录开始格式化事件
        span.add_event("Calling format_documents")
        # 记录待格式化文档数量
        span.set_attribute("input.documents_count", len(retrieved_docs))

        t = ''
        for i, doc in enumerate(retrieved_docs):
            t += f'Retrieved doc: {doc}\n'
            # 为每个处理过的文档记录事件
            span.add_event(f"processed document {i}", {"document.content": doc})

        # 文档格式化完成后标记 span 成功
        span.set_status(Status(StatusCode.OK))
    return t

def augment_prompt(query, formatted_documents):
    # 启动 span，追踪提示词增强过程
    with tracer.start_as_current_span("augment_prompt") as span:
        # 记录提示词增强开始事件
        span.add_event("Starting prompt augmentation")
        # 记录输入信息，如 query 与文档长度
        span.set_attribute("input.query", query)
        span.set_attribute("input.formatted_documents_length", len(formatted_documents))

        # 构造融合 query 与文档内容的提示词
        PROMPT = f"Answer the query: {query}.\nRelevant documents:\n{formatted_documents}"

        # 标记 span 成功
        span.set_status(Status(StatusCode.OK))
    return PROMPT

def generate(prompt):
    # 启动 span，追踪基于提示词的文本生成
    with tracer.start_as_current_span("generate") as span:
        # 记录开始生成事件
        span.add_event("Starting text generation")
        # 记录用于生成的 prompt
        span.set_attribute("input.prompt", prompt)

        # 模拟文本生成过程
        generated_text = f"Generated text for prompt {prompt}"

        # 文本生成完成后标记 span 成功
        span.set_status(Status(StatusCode.OK))
    return generated_text

def rag_pipeline(query, fail = False):
    # 启动 span，追踪整个 RAG 流水线
    with tracer.start_as_current_span("rag_pipeline") as span:
        try:
            # 步骤 1：基于 query 检索文档
            retrieved_docs = retrieve(query, fail = fail)
            # 步骤 2：格式化检索文档
            formatted_docs = format_documents(retrieved_docs)
            # 步骤 3：把 query 与相关文档组装为 prompt
            prompt = augment_prompt(query, formatted_docs)
            # 步骤 4：基于增强 prompt 生成回答
            generated_response = generate(prompt)

            # 所有步骤完成后标记 span 成功
            span.set_status(Status(StatusCode.OK))
            return generated_response
        except Exception as e:
            # 任一步骤抛错时，将 span 状态标记为错误
            span.set_status(Status(StatusCode.ERROR, str(e)))
            # 将异常继续抛出，便于外层处理
            raise

In [ ]:
# Trace 示例 1
response = rag_pipeline("This is a test query", fail = False)

{
    "name": "retrieving_documents",
    "context": {
        "trace_id": "0x6b16a0356f2905613dc5b1fedb1399da",
        "span_id": "0x9bbfc55baceb094c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0efd18693f062ad8",
    "start_time": "2026-04-03T12:18:25.096858Z",
    "end_time": "2026-04-03T12:18:25.096929Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "input.query": "This is a test query",
        "retrieval.documents.0.document.id": 0,
        "retrieval.documents.0.document.content": "retrieved doc1",
        "retrieval.documents.0.document.metadata": "Metadata for document 0",
        "retrieval.documents.1.document.id": 1,
        "retrieval.documents.1.document.content": "retrieved doc2",
        "retrieval.documents.1.document.metadata": "Metadata for document 1",
        "retrieval.documents.2.document.id": 2,
        "retrieval.documents.2.document.content": "retrieved doc3",
        "retrieval.do

In [ ]:
# Trace 示例 2
response = rag_pipeline("This is a test query", fail = True)

{
    "name": "retrieving_documents",
    "context": {
        "trace_id": "0x28a7a107fafd2a3e523b52b93d031800",
        "span_id": "0x9e2c4b90712bcd84",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x9766ffb603cad088",
    "start_time": "2026-04-03T12:18:26.808616Z",
    "end_time": "2026-04-03T12:18:26.810221Z",
    "status": {
        "status_code": "ERROR",
        "description": "ValueError: Retrieve failed for query: This is a test query"
    },
    "attributes": {
        "input.query": "This is a test query",
        "error.type": "ValueError",
        "error.message": "Retrieve failed for query: This is a test query"
    },
    "events": [
        {
            "name": "Starting retrieve",
            "timestamp": "2026-04-03T12:18:26.808630Z",
            "attributes": {}
        },
        {
            "name": "exception",
            "timestamp": "2026-04-03T12:18:26.810197Z",
            "attributes": {
                "exception.t

ValueError: Retrieve failed for query: This is a test query


<div class="alert alert-block alert-warning">
注意：第二条 trace 是故意让它失败，用于展示在生产系统中失败链路可能的表现。
</div>

如你所见，trace 的原始形态在大型系统中可能非常复杂且不易阅读，尤其当组件很多、相互依赖较强时更是如此。这正是 Phoenix 这类工具重要的原因：它们可以帮助你管理并可视化 trace，从而更高效地分析数据、定位性能问题和系统瓶颈。

<a id='3'></a>
## 3 - 使用 Phoenix 进行遥测
---

Phoenix 是一个用于简化遥测数据管理与可视化的强大工具。它可以帮助你处理复杂 trace，让分析与诊断系统问题变得更容易。借助 Phoenix，你可以监控 RAG 系统性能、识别瓶颈，并深入了解应用不同组件之间的交互方式。在本节中，你将学习如何配置 Phoenix，并利用它的能力更好地理解 RAG 系统产生的遥测数据。

In [8]:
import utils
import phoenix as px

<a id='3-1'></a>
### 3.1 启动 Phoenix 应用

运行下一个单元来启动 Phoenix 应用，它会开启本地服务并提供一个用户界面（UI）。默认访问地址是 `localhost:6006`，在调用应用后会显示该地址。由于 Coursera 环境限制，系统会提供一个替代链接。你可以点击该链接在新标签页打开 UI。

In [9]:
utils.make_url()
px.launch_app()

FOLLOW THIS URL TO OPEN THE UI: https://s172-29-69-151p6006.lab-aws-production.deeplearning.ai
🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


你应该会看到如下界面：

![Phoenix UI Screenshot](images/ui_1.png)

<a id='3-2'></a>
### 3.2 准备遥测配置

现在你将配置与 Phoenix 配套使用的遥测。由于 Phoenix 同样基于 OpenTelemetry，因此配置方式与上面非常相似。

In [ ]:
from phoenix.otel import register
from opentelemetry.trace import Status, StatusCode
phoenix_project_name = "example-rag-pipeline"

# 在 Phoenix 中，只需注册即可获得带有正确端点的 tracer provider。
endpoint="http://127.0.0.1:6006/v1/traces"
tracer_provider_phoenix = register(project_name=phoenix_project_name, endpoint = endpoint)

# 获取一个 tracer，用于手动埋点
tracer = tracer_provider_phoenix.get_tracer(__name__)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: example-rag-pipeline
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



<a id='3-3'></a>
### 3.3 使用流水线

#### 3.3.1 检索

这还是同一个 toy 检索函数。语法几乎一致，但有两个差异：

1. 现在有一个 `openinference_span_kind` 参数，你可以将其设置为 retriever。
2. 现在可以通过 `span.set_input` 设置输入。 

In [ ]:
def retrieve(query, fail=False):
    # 启动 span 追踪检索流程。现在可以传入 span kind: retriever
    with tracer.start_as_current_span("retrieving_documents", openinference_span_kind = 'retriever') as span:
        # 记录开始检索事件
        span.add_event("Starting retrieve")
        # 记录输入 query，便于可观测性
        # Phoenix 允许使用 span.set_input
        span.set_input(query)
        try:
            # 若 'fail' 为 True，模拟检索失败
            if fail:
                raise ValueError(f"Retrieve failed for query: {query}")

            # 模拟检索到的文档列表
            retrieved_docs = ['retrieved doc1', 'retrieved doc2', 'retrieved doc3']
            # 记录每个检索文档的细节
            for i, doc in enumerate(retrieved_docs):
                span.set_attribute(f"retrieval.documents.{i}.document.id", i)
                span.set_attribute(f"retrieval.documents.{i}.document.content", doc)
                span.set_attribute(f"retrieval.documents.{i}.document.metadata", f"Metadata for document {i}")
        except Exception as e:
            # 若发生异常，记录并将 span 状态标记为错误
            span.set_status(Status(StatusCode.ERROR, str(e)))
            span.set_attribute("error.type", type(e).__name__)
            span.set_attribute("error.message", str(e))
            # 将异常继续抛出，由调用方处理
            raise

        # 若无错误，标记 span 成功
        span.set_status(Status(StatusCode.OK))
        return retrieved_docs

<a id='3-4'></a>
### 3.4 Chain

Chain 是 LLM 应用中不同步骤之间的连接点。它可以把多个操作串起来，例如发起请求、或把检索器输出传递给 LLM 调用。Chain 能让流程更有组织、更易维护。

#### 3.4.1 剩余的 RAG 函数

这些函数与你之前使用的是同一套，只是现在加上了 `@tracer.chain` 装饰器。你只需要把它加在希望被追踪的函数前，就会作为 chain 被记录。如果你想做更细粒度追踪，则应像上面的 retriever 那样手动埋点。

In [12]:
@tracer.chain
def format_documents(retrieved_docs):
    t = ''
    for i, doc in enumerate(retrieved_docs):
        t += f'Retrieved doc: {doc}\n'
    return t

@tracer.chain
def augment_prompt(query, formatted_documents):
    
    # Create a prompt that combines the query and formatted documents
    PROMPT = f"Answer the query: {query}.\nRelevant documents:\n{formatted_documents}"
    return PROMPT

@tracer.chain
def generate(prompt):
    generated_text = f"Generated text for prompt {prompt}"
    return generated_text

@tracer.chain
def rag_pipeline(query, fail = False):
        # Step 1: Retrieve documents based on the query
        retrieved_docs = retrieve(query, fail = fail)
        # Step 2: Format the retrieved documents
        formatted_docs = format_documents(retrieved_docs)
        # Step 3: Augment the query with relevant documents to form a prompt
        prompt = augment_prompt(query, formatted_docs)
        # Step 4: Generate a response from the augmented prompt
        generated_response = generate(prompt)
        return generated_response

<a id='3-5'></a>
### 3.5 使用 UI 分析 Trace

现在来到更有趣的部分！运行下面单元，执行与前面相同的两次查询。然后打开 Phoenix UI 查看这些 trace 在界面中的展示方式。

In [13]:
response = rag_pipeline("This is a test query")
try:
    response = rag_pipeline("This is a test query that failed", fail = True)
except:
    pass

In [14]:
utils.make_url()

FOLLOW THIS URL TO OPEN THE UI: https://s172-29-69-151p6006.lab-aws-production.deeplearning.ai


你会看到类似如下界面，所有信息都会以更有组织的方式展示。


![Phoenix UI Screenshot](images/ui_3.png)

In [ ]:
# 重启内核，以演示启用 auto_instrument 的全新 Phoenix 配置
# 这次重启用于展示两种不同 Phoenix 配置：
# - 上一节：基础 Phoenix 追踪
# - 下一节：auto_instrument=True 下的 OpenAI 自动追踪
# 注意：在生产环境中，通常只会在应用启动时配置一次 Phoenix
utils.restart_kernel()

<a id='4'></a>
## 4 - 结合 Weaviate 的追踪与评估

---

你现在已经掌握了使用 Phoenix 做遥测的基础能力。接下来我们进入更具体的场景：使用 M4 Assignment 的 FAQ 问题，构建一个小型 RAG 流水线来回答服装商店相关 FAQ。

In [ ]:
from phoenix.otel import register
from opentelemetry.trace import Status, StatusCode
import phoenix as px
import utils
import weaviate

# 在导入 flask_app 和 weaviate_server 之前，清理相关端口进程
# 警告：重复运行该单元可能会终止当前活动内核
utils.kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])

import flask_app
import weaviate_server

Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 30] Read-only file system: '.models/models--BAAI--bge-base-en-v1.5/.no_exist/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/adapter_config.json'


 * Serving Flask app 'flask_app'
 * Debug mode: off


In [ ]:
# 清理已有 Phoenix 会话，避免项目 ID 冲突
utils.cleanup_phoenix_projects()

utils.make_url()
session = px.launch_app()

No active session to close
Phoenix cleanup completed
FOLLOW THIS URL TO OPEN THE UI: https://s172-29-69-151p6006.lab-aws-production.deeplearning.ai
🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


<a id='4-1'></a>
### 4.1 配置追踪器

配置方式与之前相同，但这里新增了 `auto_instrument` 参数。将其设为 `True` 后，可自动追踪 OpenAI 兼容的 LLM 调用！

In [ ]:
from phoenix.otel import register
import time

# 生成唯一项目名以避免冲突
phoenix_project_name = f"example-rag-pipeline-with-weaviate-{int(time.time())}"

# 在 Phoenix 中，只需注册即可获得带有正确端点的 tracer provider。
# 传入 auto_instrument=True 后，OpenAI 调用会自动被追踪
# TogetherAI 与 OpenAI 兼容
tracer_provider_phoenix = register(project_name=phoenix_project_name, endpoint="http://127.0.0.1:6006/v1/traces", auto_instrument=True)

# 获取 tracer，用于手动埋点
tracer = tracer_provider_phoenix.get_tracer(__name__)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: example-rag-pipeline-with-weaviate-1775218776
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



<a id='4-2'></a>
### 4.2 准备 Weaviate 客户端与集合


In [ ]:
# 连接 weaviate 客户端
client = weaviate.connect_to_local(port=8079, grpc_port=50050)

# 确保 FAQ 集合存在且已写入数据
utils.setup_faq_collection()

FAQ collection already exists


In [5]:
import joblib
data = joblib.load("faq.joblib")

In [ ]:
# 回顾数据结构
data[0]

{'question': 'What are your store hours?',
 'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.',
 'type': 'general information'}

In [ ]:
# 加载集合
collection = client.collections.get("Faq")

In [8]:
len(collection)

25

<a id='4-3'></a>
### 4.3 检索器

现在你将像之前一样配置一个检索器。这一次我们还会加入遥测信息，以便追踪并理解检索过程。 

In [ ]:
def retrieve(query_text, limit=5):
    # 为查询启动一个 span
    with tracer.start_as_current_span(
        "query_weaviate", openinference_span_kind="retriever"
    ) as span:
        # 设置 span 输入
        span.set_input(query_text)

        # 查询集合
        collection_name = "Faq"
        chunks = client.collections.get(collection_name)
        results = chunks.query.near_text(query=query_text, limit=limit)

        # 将检索到的文档写入 span 属性
        for i, document in enumerate(results.objects):
            span.set_attribute(f"retrieval.documents.{i}.document.id", str(document.uuid))
            span.set_attribute(f"retrieval.documents.{i}.document.metadata", str(document.metadata))
            span.set_attribute(
                f"retrieval.documents.{i}.document.content", str(document.properties)
            )  

        return results

In [ ]:
# 处理并格式化检索结果
@tracer.chain 
def format_context(results):
    context = ""
    for item in results.objects:
        properties = item.properties
        context += f"Question: {properties['question']}\n"
        context += f"Answer: {properties['answer']}\n"
    return context
     

In [ ]:
# 基于检索信息构造提示词
@tracer.chain
def create_prompt(query_text, context):
    prompt = f"""
Based on the following information, please answer the FAQ related question: "{query_text}"

Relevant FAQ (ordered by relevance):
{context}
"""
    return prompt

<a id='4-4'></a>
### 4.4 使用 `openai` 库进行 LLM 调用

由于 Phoenix 可与 OpenAI 风格系统集成，我们就采用这种方式。[together.ai](https://www.together.ai/) 恰好兼容 OpenAI！ 

In [12]:
import httpx
from openai import OpenAI, DefaultHttpxClient

In [ ]:
# 自定义传输层以跳过 SSL 校验
transport = httpx.HTTPTransport(local_address="0.0.0.0", verify=False)

# 使用自定义传输层创建 DefaultHttpxClient
http_client = DefaultHttpxClient(transport=transport, headers=utils.get_proxy_headers())

# 这里可以使用任意 OpenAI 兼容端点
llm_client = OpenAI(
    api_key = utils.get_together_key(), # 这里代理不会实际校验该值；若使用 together 官方端点请填真实 key
    base_url=utils.get_proxy_url(), # 跨平台：自动识别 Coursera、Learning Platform 或本地环境
    http_client=http_client, # 通过代理调用时用于绕过 ssl，若直连 together.ai 可移除
)

In [ ]:
# 因为 auto_instrument=True，这里无需手动追踪
def query_openai(prompt):
    response = llm_client.chat.completions.create(
        model="Qwen/Qwen3.5-9B",
        extra_body={"reasoning": False},
        messages=[
            {"role": "system", "content": "You are a helpful assistant from a customer support."},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content

In [ ]:
@tracer.chain
def rag_pipeline(query):
    # 执行检索
    retrieved_documents = retrieve(query)
    context = format_context(retrieved_documents)
    
    # 基于检索信息构造提示词
    final_prompt = create_prompt(query, context)
    
    # 发起 OpenAI 风格查询
    final_answer = query_openai(final_prompt)

    return final_answer

In [16]:
response = rag_pipeline("Can I get a refund or exchange for another shirt?")
print(response)

Hello! Yes, you can generally get a return or exchange for a shirt, subject to our policy guidelines. Here are the key details to help you proceed:

*   **Timeframe:** Returns are accepted within 30 days of delivery.
*   **Sale Items:** Please note that sale items are final sale and cannot be returned or exchanged unless stated otherwise.
*   **How to Exchange:** You can initiate an exchange through our Returns Center by selecting the item you wish to exchange and the desired


In [17]:
response = rag_pipeline("What are your working hours?")
print(response)

Our online store is open 24/7. However, our customer service team is available to assist you from 9:00 AM to 6:00 PM, Monday through Friday.


In [ ]:
# 去 Phoenix UI 查看 trace！
utils.make_url()

FOLLOW THIS URL TO OPEN THE UI: https://s172-29-69-151p6006.lab-aws-production.deeplearning.ai


继续加油！你已完成 Phoenix 遥测无评分实验！